# Test Robot Motion

Loads `uniwheel.xml` into MuJoCo and lets you drive it by hand:
- **Up / Down arrows**: `wheel_motor` speed (ground wheel — pitch axis, drives forward/back)
- **Left / Right arrows**: `reaction_wheel_motor` speed (flywheel — roll axis)

Try to catch it before it tips. Good luck — see `../../../docs/feasibility_analysis.md`
and `STAGE01_FINDINGS.md` in this folder for why this is harder than the reference
repo's 2-wheel bot (this one has a ~18mm CG offset from the wheel axis, so it
doesn't rest upright on its own).

MuJoCo API reference: https://mujoco.readthedocs.io/en/stable/APIreference/index.html


In [1]:
# Import standard libraries
from pathlib import Path
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer


In [2]:
# Settings
MJCF_PATH = Path("../../mechanical/urdf/uniwheel/uniwheel.xml")
MOTOR_SPEED_STEP = 0.1     # How much to increase/decrease motor speed by
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 50           # Number of sim loop iterations before printing sensor readings

# Actuator names (from MJCF file)
WHEEL_MOTOR = "wheel_motor"
REACTION_WHEEL_MOTOR = "reaction_wheel_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"
WHEEL_POS = "wheel_pos"
WHEEL_VEL = "wheel_vel"
REACTION_WHEEL_POS = "reaction_wheel_pos"
REACTION_WHEEL_VEL = "reaction_wheel_vel"

# Keys
KEY_BACKSPACE = 259
KEY_UP = 265
KEY_DOWN = 264
KEY_LEFT = 263
KEY_RIGHT = 262


In [3]:
# Global: speed of each motor
ctrl = {"wheel": 0.0, "reaction_wheel": 0.0}


In [4]:
def clamp(x):
    """Limit motor speed and direction to a minimum and maximum"""
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))


In [5]:
def key_callback(keycode):
    """Handle a key press"""
    # MuJoCo resets the sim with backspace, but we also need to stop the motors
    if keycode == KEY_BACKSPACE:
        ctrl["wheel"] = ctrl["reaction_wheel"] = 0.0
        return

    if keycode == KEY_UP:
        ctrl["wheel"] = clamp(ctrl["wheel"] + MOTOR_SPEED_STEP)
    elif keycode == KEY_DOWN:
        ctrl["wheel"] = clamp(ctrl["wheel"] - MOTOR_SPEED_STEP)
    elif keycode == KEY_RIGHT:
        ctrl["reaction_wheel"] = clamp(ctrl["reaction_wheel"] + MOTOR_SPEED_STEP)
    elif keycode == KEY_LEFT:
        ctrl["reaction_wheel"] = clamp(ctrl["reaction_wheel"] - MOTOR_SPEED_STEP)
    else:
        return


In [6]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data = mujoco.MjData(model)


In [7]:
# Get ID of actuators from MJCF names
wheel_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, WHEEL_MOTOR)
reaction_wheel_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, REACTION_WHEEL_MOTOR)

# Print IDs
print(f"Wheel motor ID: {wheel_motor_id}")
print(f"Reaction wheel motor ID: {reaction_wheel_motor_id}")


Wheel motor ID: 0
Reaction wheel motor ID: 1


In [1]:
# Resets simulation data to defaults
mujoco.mj_resetData(model, data)

# Launch MuJoCo simulator and GUI
steps = 0
with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
    # Define free-look camera (control with mouse)
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.08]
    viewer.cam.distance = 0.8
    viewer.cam.azimuth = 45
    viewer.cam.elevation = -25

    # Simulation loop
    while viewer.is_running():
        step_start = time.time()

        # Write the latest ctrl values into the sim
        data.ctrl[wheel_motor_id] = ctrl["wheel"]
        data.ctrl[reaction_wheel_motor_id] = ctrl["reaction_wheel"]

        # Advance simulation by one step (determined by timestep attribute in MJCF file)
        mujoco.mj_step(model, data)

        # Render the current simulation state
        viewer.sync()

        # Print the sensor readings every PRINT_EVERY steps
        steps += 1
        if steps % PRINT_EVERY == 0:
            accel = data.sensor(IMU_ACCEL).data
            gyro = data.sensor(IMU_GYRO).data
            orientation = data.sensor(IMU_ORIENTATION).data
            wheel_pos = data.sensor(WHEEL_POS).data
            wheel_vel = data.sensor(WHEEL_VEL).data
            rwheel_pos = data.sensor(REACTION_WHEEL_POS).data
            rwheel_vel = data.sensor(REACTION_WHEEL_VEL).data

            clear_output(wait=True)
            print(f"Accel:            {accel}")
            print(f"Gyro:             {gyro}")
            print(f"Orientation:      {orientation}")
            print(f"Wheel pos/vel:    {wheel_pos} / {wheel_vel}")
            print(f"React. wheel p/v: {rwheel_pos} / {rwheel_vel}")

        # Slow the simulation down so we can see the robot behave in real time
        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)


NameError: name 'mujoco' is not defined